# Day 4, Part 2b: Coding Real Abstracts with an LLM, and the Problem of Validation

**MIDAS Summer Academy**

In Part 2a you drove the Groq API and saw how prompts and parameters change the output. Here we point that tool at a real task: coding a set of **real published research abstracts** for methodological features. Then we hit the question that actually matters for research: **how would you know the codes are right?**

You will:
1. Use an LLM for **content analysis**, coding unstructured text for defined features
2. Force structured JSON output and track token usage
3. Add a genuinely interpretive judgment (`overclaims`) that even human coders disagree on
4. Confront validation **when there is no answer key**, which is the normal situation

### About this corpus

These are 30 real, open-access abstracts from behavioral and health psychology, reused under their Creative Commons licenses (attributions are in `attributions.csv`). They are messier than a textbook example on purpose: real jargon, real hedging, and no clean ground truth for the harder judgments. That last point is the whole lesson.

## Setup

Same client as Part 2a. We use `llama-3.3-70b-versatile`, which supports a strict JSON output mode, and code at `temperature=0` for reproducibility.

In [ ]:
!pip install -q groq pandas

In [ ]:
from getpass import getpass
from groq import Groq
import json, os, time
import pandas as pd

client = Groq(api_key=getpass("Paste your Groq API key: "))
MODEL = "llama-3.3-70b-versatile"
print("Client ready.")

### The coding function

We describe the five features, turn on JSON mode so the reply parses directly, and give the model a working definition of the two hard judgments (`makes_causal_claim` and `overclaims`). A definition gives the model a fair chance, but as you will see, it does not make the judgment objective.

In [ ]:
EXTRACTION_SYSTEM = (
    "You are a careful content-analysis coder for a research-methods project. "
    "You read study abstracts and record features precisely. Use only what the text "
    "states. If a feature is not stated, record null (or false for the yes/no features)."
)

EXTRACTION_PROMPT = (
    "Read the research abstract and code these features. Return a JSON object with "
    "exactly these keys:\n"
    "- 'sample_size': total number of participants as an integer, or null if not stated.\n"
    "- 'population': short description of the participants, or null if not described.\n"
    "- 'study_design': one of 'experimental' (participants randomly assigned to "
    "conditions), 'correlational' (variables measured, not manipulated), 'survey' "
    "(questionnaire or self-report only), or 'other'.\n"
    "- 'makes_causal_claim': true if the conclusion states that one variable causes, "
    "increases, reduces, or improves another; false if it reports only an association, "
    "correlation, or link.\n"
    "- 'overclaims': true if the conclusion makes a causal or prescriptive claim its "
    "design does not support; false otherwise. Guidance: an experimental study may claim "
    "causation, so overclaims is usually false there. A correlational or survey study that "
    "only reports associations does not overclaim. A correlational study that concludes one "
    "variable causes/drives/reduces another, or recommends changing one variable to affect "
    "another, does overclaim.\n\n"
    "Abstract:\n{abstract}"
)

def code_abstract(abstract_text):
    # Send one abstract; return (parsed_dict, usage).
    resp = client.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": EXTRACTION_SYSTEM},
                  {"role": "user", "content": EXTRACTION_PROMPT.format(abstract=abstract_text)}])
    return json.loads(resp.choices[0].message.content), resp.usage

## 1. Load the corpus

The corpus lives in `real_abstracts/` when you run this from the repo, and falls back to a public copy for Colab.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/elleobrien/MIDAS_summer_academy_student/main/day04/real_abstracts"

def load_corpus(name):
    local = f"real_abstracts/{name}"
    return pd.read_csv(local if os.path.exists(local) else f"{DATA_URL}/{name}")

abstracts = load_corpus("abstracts.csv")
example = abstracts.iloc[0]["abstract"]
print(f"Loaded {len(abstracts)} real abstracts.\n")
print(abstracts.iloc[0]["abstract_id"])
print(example)

### The five features, from fact to judgment

| Feature | What we record | Difficulty |
|---|---|---|
| `sample_size` | number of participants | Easy: a fact in the text |
| `population` | who the participants were | Medium |
| `study_design` | experimental / correlational / survey / other | Medium, needs interpretation |
| `makes_causal_claim` | does the conclusion assert causation vs. association | Hard: a judgment |
| `overclaims` | does the causal/actionable claim exceed what the design supports | Hardest: a deep judgment |

`sample_size` is a fact the model just has to find. `overclaims` requires reasoning about the study's design *and* the strength of its conclusion together, and reasonable human coders genuinely disagree on it. A working rubric: an experiment may claim causation; a correlational study that only reports associations does not overclaim; a correlational study that concludes one thing *causes* another, or that you should change X to move Y, does. "Predicts" is borderline. Hedges ("longitudinal work is needed") pull the other way, and a single abstract can hedge in one sentence and overclaim in the next.

## 2. Code one abstract

Test on one before running a batch.

In [ ]:
data, usage = code_abstract(example)
print(json.dumps(data, indent=2))
print("\nTokens:", usage.prompt_tokens, "in +", usage.completion_tokens, "out")

The model confidently assigned an `overclaims` value. Do you agree with it? Read the abstract and decide. There is no key in this corpus that can tell you who is right, and that is deliberate.

## 3. Code the whole corpus

One abstract per call (each is independent, and one failure won't sink the rest), with a short pause to respect the free-tier limit.

In [ ]:
N_PROCESS = 30
subset = abstracts.head(N_PROCESS)

records, errors = [], []
tokens_in = tokens_out = 0
for row in subset.itertuples():
    print(f"Coding {row.abstract_id}...", end=" ")
    try:
        d, u = code_abstract(row.abstract)
        d["abstract_id"] = row.abstract_id
        records.append(d)
        tokens_in += u.prompt_tokens; tokens_out += u.completion_tokens
        print("ok")
    except Exception as e:
        errors.append({"abstract_id": row.abstract_id, "error": str(e)})
        print(f"error: {e}")
    time.sleep(1)
print(f"\nCoded {len(records)}, errors {len(errors)}")

### What did that cost?

> The price per token is an **assumption** for illustration. Check Groq's current pricing before quoting a real number.

In [ ]:
total = tokens_in + tokens_out
print(f"{len(records)} abstracts: {tokens_in} in + {tokens_out} out = {total} tokens")
PRICE_IN_PER_M, PRICE_OUT_PER_M = 0.59, 0.79   # assumed $/million tokens
cost = tokens_in/1e6*PRICE_IN_PER_M + tokens_out/1e6*PRICE_OUT_PER_M
print(f"Estimated cost: ${cost:.4f}")
per = cost/len(records)
for size in [1000, 50_000]:
    print(f"  ~{size:>6} abstracts: ${per*size:7.2f}, about {size*1.2/60:.0f} min")

## 4. Assemble the results

In [ ]:
results = pd.DataFrame(records)[
    ["abstract_id", "sample_size", "population", "study_design",
     "makes_causal_claim", "overclaims"]
]
results

## 5. Validation, with no answer key

This is the real situation for most research text: you have the model's codes and **no gold standard to compare against.** So what can you actually check?

### Rung 1: cheap automatic checks (run on everything, no key needed)

- **Structural:** is `study_design` one of the allowed values, and are the types right?
- **Grounding:** if the model reported a `sample_size`, does that number appear in the abstract? This catches invented numbers.

In [ ]:
ALLOWED = {"experimental", "correlational", "survey", "other"}

def structural_ok(d):
    if d.get("study_design") not in ALLOWED:
        return False
    ss = d.get("sample_size")
    if ss is not None and not (isinstance(ss, int) and ss > 0):
        return False
    return isinstance(d.get("makes_causal_claim"), bool) and isinstance(d.get("overclaims"), bool)

def sample_size_grounded(d, text):
    ss = d.get("sample_size")
    if ss is None:
        return True
    return str(ss) in text

text_by_id = dict(zip(abstracts["abstract_id"], abstracts["abstract"]))
print("structural OK       :", sum(structural_ok(r) for r in records), "/", len(records))
print("sample_size grounded:", sum(sample_size_grounded(r, text_by_id[r["abstract_id"]]) for r in records), "/", len(records))

These checks confirm the output is well-formed and that reported sample sizes are real numbers from the text. They tell you **nothing** about whether `makes_causal_claim` or `overclaims` is correct. There is no field to ground a judgment against, and no key to score it with.

### Rung 2: you are the coder (error analysis)

The only way to evaluate the judgments is to read the abstracts and code them yourself. Here are the correlational studies the model flagged, or did not flag, as overclaiming. Read a few, decide what you would code, and compare.

In [ ]:
corr = [r for r in records if r.get("study_design") in ("correlational", "survey")]
print(f"{len(corr)} correlational/survey studies coded.\n")
for r in corr[:6]:
    print(f"[{r['abstract_id']}]  overclaims = {r['overclaims']}   (causal_claim = {r['makes_causal_claim']})")
    print(text_by_id[r["abstract_id"]])
    print()

You will likely disagree with the model on some of these, and if you compare notes with the person next to you, you will disagree with *each other*. That is not a failure of the exercise; it is the actual state of affairs. `overclaims` has no objective answer, only defensible human judgments.

This is what validation of an interpretive code really requires, and why it is expensive:
- Multiple people code the same items independently.
- You measure how often they agree (inter-rater reliability, e.g. Cohen's kappa).
- You adjudicate disagreements and write a sharper rubric.
- Only once humans agree with each other do you have a standard to hold the model to.

We are naming that frontier, not building it. A few principles worth carrying out, from Hamel Husain's [evals FAQ](https://hamel.dev/blog/posts/evals-faq/):
- **Look at your data.** You find failure modes by reading outputs, not in the abstract.
- **Prefer binary judgments** over 1-to-5 scores.
- **A clean-looking result is not a validated one.** Structural checks passing tells you nothing about the judgments.
- **An automated coder is not trustworthy until it agrees with humans who agree with each other.**

## 6. The label depends on the rubric, not just the text

Here is a concrete way to feel how soft `overclaims` is: change the definition and watch the labels move. Below, a stricter rubric that also counts "predicts/predictor" as overclaiming. Re-code the correlational studies and count how many labels flip.

This is the exercise. There is no "right" number, which is exactly the point: the code is a function of your rubric, and choosing the rubric is a human decision you cannot outsource to the model.

In [ ]:
# YOUR CODE HERE
# STRICTER_SYSTEM = EXTRACTION_SYSTEM + (
#     " Treat 'predicts' or 'predictor' language in a correlational study as overclaiming too."
# )
#
# def code_stricter(text):
#     resp = client.chat.completions.create(
#         model=MODEL, temperature=0, response_format={"type": "json_object"},
#         messages=[{"role": "system", "content": STRICTER_SYSTEM},
#                   {"role": "user", "content": EXTRACTION_PROMPT.format(abstract=text)}])
#     return json.loads(resp.choices[0].message.content)
#
# flips = 0
# for r in corr[:8]:
#     new = code_stricter(text_by_id[r["abstract_id"]])
#     if bool(new["overclaims"]) != bool(r["overclaims"]):
#         flips += 1
#         print(r["abstract_id"], r["overclaims"], "->", new["overclaims"])
#     time.sleep(1)
# print("labels that flipped under the stricter rubric:", flips)

## When to reach for an LLM

| Use an LLM when... | Use traditional methods when... |
|---|---|
| The text is unstructured and varies a lot | The format is fixed and predictable (use a parser) |
| The feature needs interpretation | The feature is an exact pattern (use regex) |
| You have dozens to thousands of documents | You have millions (cost and speed) |

Whichever you use: test on a few first, define a strict output format, handle errors, track tokens, save raw responses, and remember that a code you cannot validate is a hypothesis, not a result.

## Summary

You coded 30 real abstracts for five features, from a simple fact (`sample_size`) to a deep judgment (`overclaims`), forced structured output, and tracked cost. The point of using real text with no answer key: **the model produces confident labels for the hardest judgments, cheap checks cannot tell you whether they are right, and building something that can requires human coders who first agree with each other.** That labor, not the API call, is the real work of trustworthy content analysis.

### Reflection
- For your own research text, which features are facts and which are judgments like `overclaims`?
- How many people would you need, and how would you measure their agreement, before trusting an automated coder?
- What would you do with the abstracts where even careful coders disagree?

*Abstracts are real open-access papers reused under their Creative Commons licenses; see `attributions.csv` for full credit to each source.*